# Day 1 · Brief → Bubble Diagram

**Idea:** type a design brief (e.g. *"2-bed flat with a home studio"*). A **local LLM** (Ollama) turns it into a room programme (rooms, areas, zones, adjacencies) as structured JSON, and **networkx + matplotlib** draw it as an architectural bubble diagram, with circles sized by area.

**Stack:** Python · Ollama (`qwen2.5:14b`, runs offline on the Mac) · pydantic · networkx · matplotlib

**Done when:** one brief → one clean diagram image.

```
brief (text) ──► LLM (Ollama) ──► JSON programme ──► clean + connect ──► graph ──► bubble diagram (PNG)
```

> Why a notebook? The LLM call is the slow part (10–30 s). Here it runs **once** and is stored in `program`, so you can re-run only the drawing cells while tweaking the visuals.

## Step 0 · Setup

One-time, in the terminal (already done):
```bash
ollama pull qwen2.5:14b
python3 -m venv .venv && source .venv/bin/activate
```
In VS Code, pick the **.venv** kernel (top-right *Select Kernel*). If it asks to install `ipykernel`, say yes. The cell below installs the libraries into whichever kernel is running.

In [ ]:
%pip install -q ollama networkx matplotlib

## Step 1 · Define the data shape

Before asking the LLM anything, decide **exactly what we want back**. These pydantic classes are the "contract":

- **Room**: a name, an area in m², and a zone (`public`, `private`, `service`, `work`). The zone becomes the colour.
- **Link**: two room names and how strongly they should be adjacent (`strong` / `weak`).
- **Program**: a list of rooms plus a list of links.

`Literal[...]` restricts a field to a fixed set of values, so the model can't invent a zone like `"chill"`.

In [ ]:
from pydantic import BaseModel
from typing import Literal

class Room(BaseModel):
    name: str
    area_m2: float
    zone: Literal["public", "private", "service", "work"]

class Link(BaseModel):
    a: str
    b: str
    strength: Literal["strong", "weak"]

class Program(BaseModel):
    rooms: list[Room]
    adjacencies: list[Link]

## Step 2 · Ask the local LLM

- `SYSTEM` is the instruction that tells the model to act as an architectural programmer.
- `format=Program.model_json_schema()` passes our data shape to Ollama, which **forces** the reply to be valid JSON in exactly that shape (structured output).
- `temperature: 0` makes the answer as deterministic as possible.
- `Program.model_validate_json(...)` turns the JSON text back into Python objects and checks it.

This is the only slow cell. Run it once per brief.

In [ ]:
import ollama

SYSTEM = """You are an architectural programmer. Given a brief, list the rooms
with realistic areas in m², a zone for each, and which rooms should be adjacent.
Use the exact room names in the adjacencies. Include circulation (hall/corridor).
Every room must connect to at least one other room, and the hall must connect to
the living room, so the diagram is one connected graph."""

def get_program(brief: str) -> Program:
    r = ollama.chat(
        model="qwen2.5:14b",
        messages=[{"role": "system", "content": SYSTEM},
                  {"role": "user", "content": brief}],
        format=Program.model_json_schema(),
        options={"temperature": 0},
    )
    return Program.model_validate_json(r.message.content)

In [ ]:
BRIEF = "2-bed flat with a home studio"

program = get_program(BRIEF)

for room in program.rooms:
    print(f"{room.name:<20} {room.area_m2:>5.0f} m²   {room.zone}")
print()
for link in program.adjacencies:
    print(f"{link.a} ↔ {link.b}  ({link.strength})")

## Step 3 · Clean the data

LLMs sometimes make small mistakes, like linking to a room name that doesn't exist ("Hallway" vs "Hall/Corridor") or linking a room to itself. `clean` quietly drops those links so the graph doesn't crash.

In [ ]:
def clean(p: Program) -> Program:
    names = {r.name for r in p.rooms}
    p.adjacencies = [l for l in p.adjacencies
                     if l.a in names and l.b in names and l.a != l.b]
    return p

## Step 4 · Build the graph (and make sure it's connected)

- **`to_graph`**: each room becomes a **node** (storing its area and zone), and each adjacency becomes an **edge**. Strong links get `weight=3` and weak ones `weight=1`, so the layout pulls strongly-linked rooms closer together.
- **`connect`**: in the first test the model ignored "make it one connected graph", and the bedrooms floated off on their own. **Lesson: don't trust the prompt for hard rules; enforce them in code.** `connect` finds the hall (or the biggest room if there's no hall) and links any isolated cluster to it.

In [ ]:
import networkx as nx

def to_graph(p: Program) -> nx.Graph:
    G = nx.Graph()
    for r in p.rooms:
        G.add_node(r.name, area=r.area_m2, zone=r.zone)
    for l in p.adjacencies:
        G.add_edge(l.a, l.b, weight=3 if l.strength == "strong" else 1)
    return G

def connect(G: nx.Graph) -> nx.Graph:
    hub = next((n for n in G if any(w in n.lower() for w in ("hall", "corridor", "circulation"))),
               max(G, key=lambda n: G.nodes[n]["area"]))
    for comp in list(nx.connected_components(G)):
        if hub not in comp:
            biggest = max(comp, key=lambda n: G.nodes[n]["area"])
            G.add_edge(hub, biggest, weight=1)
    return G

## Step 5 · Draw the bubble diagram

- **Layout:** `spring_layout` treats edges like springs, so linked rooms pull together and unlinked ones push apart. `k` controls spacing (raise it if circles overlap), and `seed=42` keeps the layout the same on every run.
- **Size:** matplotlib's `node_size` sets the circle's *area*, so `area_m2 × 300` keeps bubbles proportional to the real m².
- **Colour = zone**, explained by the legend.
- **Lines:** solid and thick for strong adjacency, dashed for weak. Nodes are drawn *after* edges with `alpha=1`, so the circles hide the lines behind them.
- The image is saved as a PNG at 200 dpi and also shown below the cell.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

COLORS = {"public": "#F4A261", "private": "#2A9D8F",
          "service": "#B8B8B8", "work": "#E76F51"}

def draw(G: nx.Graph, brief: str, out: str = "diagram.png", k: float = 1.2, scale: float = 300):
    pos = nx.spring_layout(G, weight="weight", seed=42, k=k)
    sizes = [G.nodes[n]["area"] * scale for n in G]
    colors = [COLORS[G.nodes[n]["zone"]] for n in G]
    strong = [(u, v) for u, v, d in G.edges(data=True) if d["weight"] == 3]
    weak = [(u, v) for u, v, d in G.edges(data=True) if d["weight"] == 1]

    fig, ax = plt.subplots(figsize=(8, 8))
    nx.draw_networkx_edges(G, pos, edgelist=strong, width=2.5, edge_color="#333")
    nx.draw_networkx_edges(G, pos, edgelist=weak, width=1, style="dashed", edge_color="#999")
    nx.draw_networkx_nodes(G, pos, node_size=sizes, node_color=colors, alpha=1)
    labels = {n: f"{n}\n{G.nodes[n]['area']:.0f} m²" for n in G}
    nx.draw_networkx_labels(G, pos, labels, font_size=8)
    ax.set_title(brief, fontsize=12)
    ax.axis("off")
    ax.legend(handles=[Patch(color=c, label=z) for z, c in COLORS.items()],
              loc="lower left", frameon=False)
    plt.savefig(out, dpi=200, bbox_inches="tight", facecolor="white")
    plt.show()

## Step 6 · Run it

Clean, then build the graph, then connect it, then draw. Because `program` is already stored from Step 2, you can re-run **just this cell** while you tweak `k` or `scale`. There's no need to wait for the LLM again.

In [ ]:
G = connect(to_graph(clean(program)))
draw(G, BRIEF, k=1.2, scale=300)

## Step 7 · Batch a few briefs for the post

Runs the whole pipeline on several briefs and saves `diagram_1.png`, `diagram_2.png`, and so on. Pick the best one for Instagram.

In [ ]:
briefs = [
    "2-bed flat with a home studio",
    "small café with a coffee roastery",
    "co-living house for 6 artists",
]

for i, b in enumerate(briefs, 1):
    p = get_program(b)
    draw(connect(to_graph(clean(p))), b, out=f"diagram_{i}.png")

## Learnings · Day 1

- **Structured output** (`format=` + a pydantic schema) turns an LLM from a chatbot into a reliable data source.
- **Prompts are suggestions, code is law.** The model ignored "make it connected", so `connect()` enforces it.
- A notebook separates the **slow step** (the LLM) from the **fast iteration** (the drawing).

## What I'd do with more time (but didn't)

- Better areas (the LLM undersizes rooms). Add typical minimums to the prompt or check them in code.
- Stop circles from overlapping (a collision-aware layout).
- Export to Grasshopper / Rhino as circles plus lines, then turn the bubbles into a plan.
- A small web UI (Gradio / Streamlit) so anyone can type a brief.